In [2]:
pip install sentence-transformers


Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install pandas numpy faiss-cpu sentence-transformers geopy

Note: you may need to restart the kernel to use updated packages.


In [8]:
# domus_etl_faiss.py
# Full ETL + FAISS for Domus TA MatchBot using existing household feature code

import pandas as pd
import numpy as np
import ast
import os
from pathlib import Path
from sentence_transformers import SentenceTransformer
import faiss
from geopy.distance import geodesic


# -------------------- LOAD DATA --------------------
df_household = pd.read_excel("TA_data_Complete.xlsx")
df_household.head()

,eaorta,offerplacementpending,transferlist,rentaccountnumber,rentaccountbalance,hbreference,applicationreference,mainapplicantage,cohort,bedneed,...,hhcode,ks123,ks45,zone,L3S,flag,newhhcode,mnth,monthly,household
0,TA,N,N,1.119065e+09,"£1,176.68",70076785.0,100036,37,2,3,...,300385P00000000YY,1.0,1.0,B0,435,NaN,N300435B0,1740,1740,Applicant reference number is 100036. Bedrooms...
1,TA,N,N,1.096273e+09,£0.00,70094136.0,100110,33,2,2,...,200322P00000000YY,1.0,NaN,A1,372,NaN,N200372A1,1488,1488,Applicant reference number is 100110. Bedrooms...
2,TA,N,N,1.090911e+09,£0.00,70106646.0,100127,55,2,2,...,200385P00000000YY,NaN,1.0,B0,435,NaN,N200435B0,1740,1740,Applicant reference number is 100127. Bedrooms...
3,TA,N,N,1.100364e+09,-£436.95,70107177.0,100177,60,2,1,...,100276P00000000YY,NaN,NaN,C0,326,NaN,N100326C0,1304,1304,Applicant reference number is 100177. Bedrooms...
4,TA,N,N,1.117318e+09,"-£1,782.31",70106122.0,100212,46,2,2,...,200385P00000000YY,1.0,1.0,B0,435,NaN,N200435B0,1740,1740,Applicant reference number is 100212. Bedrooms...


In [20]:
# -------------------- LOAD DATA --------------------
household = pd.read_excel( "TA_data_Complete.xlsx")
property_df = pd.read_csv( "rightmove_with_postcodes.csv")


In [22]:
# -------------------- CLEAN HOUSEHOLD DATA --------------------
household_df = household.rename(columns={
    'applicationreference': 'Reference ID',
    'bedneed': 'Bedroom Need',
    'grossweeklyrent': 'Rent Limit',
    'postcode': 'Postcode',
    'borough': 'Zone',
    'expensiveflag': 'Flag',
    'newhhcode': 'Household Feature Code'
})

household_df['Reference ID'] = household_df['Reference ID'].fillna(0).astype(str)
household_df['Bedroom Need'] = household_df['Bedroom Need'].fillna(0).astype(int)
household_df['Rent Limit'] = pd.to_numeric(household_df['Rent Limit'], errors='coerce').fillna(0).astype(int)
household_df['Zone'] = household_df['Zone'].fillna("A1")
household_df['Flag'] = household_df['Flag'].apply(lambda x: 1 if str(x).strip().upper() == 'Y' else 0)
household_df['Extra Room'] = 0  # default

def create_household_prompt(row):
    extra = "with an extra room" if row['Extra Room'] else "with no extra room"
    need = "with special needs" if row['Flag'] else "no special needs"
    return f"Household needs {row['Bedroom Need']} bedroom(s) in zone {row['Zone']} with budget £{row['Rent Limit']} weekly, {extra}, and {need}."
household_df['match_prompt'] = household_df.apply(create_household_prompt, axis=1)

# -------------------- CLEAN PROPERTY DATA --------------------
def extract_rent(r):
    try:
        return float(r) / 4.33  # monthly to weekly
    except:
        return 0.0

property_df.rename(columns={
    'Bedrooms': 'bedrooms',
    'Bathrooms': 'bathrooms',
    'Postcode': 'postcode',
    'Address': 'address',
    'URL': 'url'
}, inplace=True)

property_df['rent'] = property_df['Rent'].apply(extract_rent).round().astype(int)
property_df['bedrooms'] = property_df['bedrooms'].fillna(0).astype(int)
property_df['bathrooms'] = property_df['bathrooms'].fillna(0).astype(int)
property_df['extra_room'] = 0
property_df['flag'] = 0

newham_hospital_coords = (51.5205, 0.0362)
def assign_zone(lat, lon):
    try:
        dist = geodesic((lat, lon), newham_hospital_coords).km
        return 'A1' if dist <= 5 else 'A2' if dist <= 10 else 'B0' if dist <= 20 else 'C0'
    except:
        return 'C0'
property_df['zone'] = property_df.apply(lambda row: assign_zone(row['Latitude'], row['Longitude']), axis=1)

def generate_property_feature_code(row):
    return f"N{row['bedrooms']}{row['extra_room']}{int(row['rent']):04d}{row['zone']}{row['flag']}"

def create_property_prompt(row):
    extra = "with an extra room" if row['extra_room'] else "with no extra room"
    need = "suitable for special needs" if row['flag'] else "not suitable for special needs"
    return f"{row['bedrooms']}-bedroom property in zone {row['zone']} for £{row['rent']} per week, {extra}, and {need}."

property_df['Property Feature Code'] = property_df.apply(generate_property_feature_code, axis=1)
property_df['match_prompt'] = property_df.apply(create_property_prompt, axis=1)

In [28]:
# -------------------- EMBEDDINGS + FAISS --------------------
model = SentenceTransformer("all-MiniLM-L6-v2")

print("✅ Embedding household prompts...")
household_embeddings = model.encode(household_df['match_prompt'].tolist(), show_progress_bar=True)

print("✅ Embedding property prompts...")
property_embeddings = model.encode(property_df['match_prompt'].tolist(), show_progress_bar=True)

household_index = faiss.IndexFlatL2(household_embeddings.shape[1])
property_index = faiss.IndexFlatL2(property_embeddings.shape[1])

household_index.add(np.array(household_embeddings))
property_index.add(np.array(property_embeddings))

# -------------------- SAVE EVERYTHING --------------------
household_df.to_csv("household_vectors.csv", index=False)
property_df.to_csv("property_vectors.csv", index=False)
faiss.write_index(household_index, "household_index.faiss")
faiss.write_index(property_index, "property_index.faiss")

print("🎉 All vector DBs and cleaned CSVs saved" )


✅ Embedding household prompts...


Batches:   0%|          | 0/223 [00:00<?, ?it/s]

✅ Embedding property prompts...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

🎉 All vector DBs and cleaned CSVs saved


In [30]:
import os
import shutil
from pathlib import Path

# Define the source (current working dir) and the destination
current_dir = Path.cwd()
target_dir = Path.home() / "Downloads" / "ta_matchbot_modified_final"
target_dir.mkdir(parents=True, exist_ok=True)

# List of files to move
files_to_move = [
    "household_vectors.csv",
    "property_vectors.csv",
    "household_index.faiss",
    "property_index.faiss"
]

# Move files to target directory
for filename in files_to_move:
    src = current_dir / filename
    dst = target_dir / filename
    if src.exists():
        shutil.move(str(src), str(dst))
        print(f"✅ Moved {filename} to {dst}")
    else:
        print(f"⚠️ File not found: {filename}")


✅ Moved household_vectors.csv to /Users/admin/Downloads/ta_matchbot_modified_final/household_vectors.csv
✅ Moved property_vectors.csv to /Users/admin/Downloads/ta_matchbot_modified_final/property_vectors.csv
✅ Moved household_index.faiss to /Users/admin/Downloads/ta_matchbot_modified_final/household_index.faiss
✅ Moved property_index.faiss to /Users/admin/Downloads/ta_matchbot_modified_final/property_index.faiss
